# Executorch Implementation

In [2]:
import torch

from model_compression.src.converters.to_executorch import convert_quantized_to_edge_pte, convert_to_executorch_program

from model_compression.src.utils import load_data
from model_compression.src.quantization.utils.model_setup import quantization_mode
from model_compression.src.utils.model_setup import setup_model
from model_compression.src.quantization.core.quantize import quantize_pytorch_model
from model_compression.src.utils.logging_setup import configure_logging

In [3]:
configure_logging(True)

### Load model

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pretrained_weights = f"models/SkinCancer/Quantized/mobilenet_v2_qat_kd.pth"
batch_size = 32
dataloaders = load_data(dataset="SkinCancer", batch_size=batch_size)
num_classes = len(dataloaders["train"].dataset.classes)
model = setup_model(model_name="mobilenet_v2", pretrained_weights=None, num_classes=num_classes)

# model.eval()
example_inputs = next(iter(dataloaders["train"]))[0].to("cpu")
# exported_model = capture_pre_autograd_graph(model, (example_inputs,))

student_model = quantization_mode(model, "export", example_inputs=(example_inputs,)).to(device)

# quantized_model = quantize_pytorch_export_model(student_model, None)

# Now load the state dict.
state_dict = torch.load(pretrained_weights, weights_only=True, map_location="cpu")
student_model.load_state_dict(state_dict)

quantized_model = quantize_pytorch_model(student_model, "export", None)

# Set to eval mode.
# torch.ao.quantization.move_exported_model_to_eval(quantized_model)

[INFO 2025-04-24 17:40:22,079 preprocessing.py:17] Train dataset size: 29322
[INFO 2025-04-24 17:40:22,080 preprocessing.py:26] Class distribution for train dataset:
[INFO 2025-04-24 17:40:22,080 preprocessing.py:28]   Class 'Actinic keratoses': 693 samples
[INFO 2025-04-24 17:40:22,080 preprocessing.py:28]   Class 'Basal cell carcinoma': 2658 samples
[INFO 2025-04-24 17:40:22,080 preprocessing.py:28]   Class 'Benign keratosis-like lesions': 2099 samples
[INFO 2025-04-24 17:40:22,081 preprocessing.py:28]   Class 'Chickenpox': 900 samples
[INFO 2025-04-24 17:40:22,081 preprocessing.py:28]   Class 'Cowpox': 792 samples
[INFO 2025-04-24 17:40:22,081 preprocessing.py:28]   Class 'Dermatofibroma': 191 samples
[INFO 2025-04-24 17:40:22,081 preprocessing.py:28]   Class 'HFMD': 1932 samples
[INFO 2025-04-24 17:40:22,081 preprocessing.py:28]   Class 'Healthy': 1368 samples
[INFO 2025-04-24 17:40:22,081 preprocessing.py:28]   Class 'Measles': 660 samples
[INFO 2025-04-24 17:40:22,082 preprocessi

Model prepared using Export Mode QAT.


/home/jacob-delgado/anaconda3/envs/executorch/lib/python3.10/site-packages/torch/fx/graph.py:1199: UserWarning: erase_node(batch_norm_104) on an already erased node
  warnings.warn(f"erase_node({to_erase}) on an already erased node")
/home/jacob-delgado/anaconda3/envs/executorch/lib/python3.10/site-packages/torch/fx/graph.py:1199: UserWarning: erase_node(batch_norm_105) on an already erased node
  warnings.warn(f"erase_node({to_erase}) on an already erased node")
/home/jacob-delgado/anaconda3/envs/executorch/lib/python3.10/site-packages/torch/fx/graph.py:1199: UserWarning: erase_node(batch_norm_106) on an already erased node
  warnings.warn(f"erase_node({to_erase}) on an already erased node")
/home/jacob-delgado/anaconda3/envs/executorch/lib/python3.10/site-packages/torch/fx/graph.py:1199: UserWarning: erase_node(batch_norm_107) on an already erased node
  warnings.warn(f"erase_node({to_erase}) on an already erased node")
/home/jacob-delgado/anaconda3/envs/executorch/lib/python3.10/sit

### Convert to ExecuTorch Program

In [6]:
convert_to_executorch_program(quantized_model, (example_inputs,), verbose=False)
convert_quantized_to_edge_pte(quantized_model, (example_inputs,))

[INFO 2025-04-24 17:40:59,584 to_executorch.py:35] Saved exported program to model.pte
[INFO 2025-04-24 17:40:59,584 to_executorch.py:113] ExecuTorch program saved as model.pte (2.31 MB)
[WARNING 2025-04-24 17:40:59,584 to_executorch.py:149] No quantized ops registered; loading kernels now.
[INFO 2025-04-24 17:41:00,286 utils.py:50] Core ATen graph:
graph():
    %p_getattr_l__self___classifier___1___bias : [num_users=1] = placeholder[target=p_getattr_l__self___classifier___1___bias]
    %p_getattr_getattr_l__self___features___0_____0___weight_bias : [num_users=1] = placeholder[target=p_getattr_getattr_l__self___features___0_____0___weight_bias]
    %p_getattr_getattr_getattr_l__self___features___1___conv___0_____0___weight_bias : [num_users=1] = placeholder[target=p_getattr_getattr_getattr_l__self___features___1___conv___0_____0___weight_bias]
    %p_getattr_getattr_l__self___features___1___conv___1___weight_bias : [num_users=1] = placeholder[target=p_getattr_getattr_l__self___features

'quantized_model.pte'